# Schema injection: JSON Schema, Pydantic, or a dataclass

nfield takes the schema you already have. Hand it a JSON Schema dict, a Pydantic model, or a
plain dataclass, and you get back the same nested JSON either way. This notebook runs all three
against one document so you can see they line up.

## Setup

Install the library with the Groq provider, then set your API key.

```bash
pip install "nfield[groq]"
export GROQ_API_KEY="gsk_..."
```

In [1]:
import os

from nfield import nfield

assert os.environ.get("GROQ_API_KEY"), "set GROQ_API_KEY first"

MODEL = "groq/llama-3.3-70b-versatile"

document = """
INVOICE #4471
Vendor: Acme Corporation
Total Due: 1284.50 USD
Date: 2026-01-15
Status: PAID
"""

## 1. A JSON Schema dict

The plain-dict form. No extra dependency, and it matches the JSON you already write for other
tools.

In [2]:
schema = {
    "type": "object",
    "properties": {
        "vendor": {"type": "string"},
        "invoice_number": {"type": "string"},
        "total": {"type": "number"},
        "currency": {"type": "string"},
        "paid": {"type": "boolean"},
    },
    "required": ["vendor", "total"],
}

result = nfield(document, schema, MODEL)
result.data

{'vendor': 'Acme Corporation',
 'invoice_number': '4471',
 'total': 1284.5,
 'currency': 'USD',
 'paid': True}

## 2. A Pydantic model

If your app already speaks Pydantic, pass the model class straight in. You keep your field types
and validators, and the result still comes back as a dict.

In [3]:
from pydantic import BaseModel


class Invoice(BaseModel):
    vendor: str
    invoice_number: str
    total: float
    currency: str
    paid: bool


result = nfield(document, Invoice, MODEL)
result.data

{'vendor': 'Acme Corporation',
 'invoice_number': '4471',
 'total': 1284.5,
 'currency': 'USD',
 'paid': True}

## 3. A dataclass

A standard-library dataclass works too, so you can stay dependency-free and still get typed
fields.

In [4]:
from dataclasses import dataclass


@dataclass
class InvoiceDataclass:
    vendor: str
    invoice_number: str
    total: float
    currency: str
    paid: bool


result = nfield(document, InvoiceDataclass, MODEL)
result.data

{'vendor': 'Acme Corporation',
 'invoice_number': '#4471',
 'total': 1284.5,
 'currency': 'USD',
 'paid': True}

## Same shape, whichever you pick

All three return the same nested JSON. Choose by what your codebase already uses:

- **JSON Schema dict** when the schema comes from a file, an API, or another tool.
- **Pydantic model** when you want validation and types you already maintain.
- **dataclass** when you want types with nothing extra to install.